# Split the FP homologs dataset

Last run top to bottom on Apr 30, 2019.

In [1]:
!pip install Bio

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.9 MB/s eta 0:00:00


In [2]:
!pip install Levenshtein

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.1/174.1 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 60.1 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import sys
import os
import random

import numpy as np
import pandas as pd
from sklearn.manifold import MDS
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

#sys.path.append('../common')
py_file_location = "/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/"
sys.path.append(os.path.abspath(py_file_location))

import data_io_utils
import paths
import utils
import constants

%reload_ext autoreload
%autoreload 2

The module path: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common
S3 data root: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3
EVO: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/evotuning_checkpoints
RANDOM: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/evotuning_checkpoints/1900_weights_random


## Seeds for reproducibility

In [6]:
np.random.seed(1)
random.seed(1)

## Load the FP homologs dataset. 

In [7]:
paths.FP_HOMOLOGS_DATA_FILE

'/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/Exp9_all_ex_lasers_FL2_inferred_brightness_parents_decomposed_tts.csv'

In [8]:
#data_io_utils.sync_s3_path_to_local(paths.FP_HOMOLOGS_DATA_FILE, is_single_file=True)

# MD5 from A049_common in mlpe-gfp-pilot repo
data_io_utils.verify_file_md5_checksum(paths.FP_HOMOLOGS_DATA_FILE, '560f032f77ece074871f210f522d8955')

df = pd.read_csv(paths.FP_HOMOLOGS_DATA_FILE)
df.head()

Verify MD5 method
Generate MD5 method
file_path: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/Exp9_all_ex_lasers_FL2_inferred_brightness_parents_decomposed_tts.csv


,quantitative_function,seq,inferred_parents,num_effective_parents,frac_seq_explained,parent_contribution_mat,tts
0,1.434310,MSKGEALFSGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,"[['Topaz'], ['moxCerulean3', 'rsEGFP2', 'SBFP2...",3,[0.69327732 0.13445378 0.1302521 ],[[0 0 0 ... 0 0 0]\n [0 0 0 ... 1 1 1]\n [0 0 ...,train
1,1.269852,MSKGAELFTGIVPILIELNGDVNGHKFSVSGEGEGDADYGKLTLKF...,"[['hfriFP', 'Azurite'], ['aceGFP'], ['Enhanced...",3,[0.65546219 0.16806723 0.17647058],[[0 0 0 ... 0 0 0]\n [0 0 0 ... 0 0 0]\n [1 1 ...,train
2,0.726303,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,"[['BFPsol'], ['GFPxm191uv', 'OFPxm'], ['TagGFP...",4,[0.59663866 0.15126051 0.14285714 0.04201681],[[1 1 1 ... 1 1 1]\n [0 0 0 ... 0 0 0]\n [0 0 ...,train
3,0.792929,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATNGKLTLKF...,"[['EGFP'], ['CFP4', 'mEmerald'], ['TagGFP', 'm...",3,[0.42857144 0.34453781 0.18907562],[[1 1 1 ... 0 0 0]\n [0 0 0 ... 1 1 1]\n [1 1 ...,train
4,1.709170,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLEIKF...,"[['Sapphire', 'mEmerald'], ['EBFP1.2'], ['mT-S...",4,[0.44117648 0.23529412 0.21848739 0.06302521],[[1 1 1 ... 0 0 0]\n [1 1 1 ... 0 0 0]\n [0 0 ...,train


### Master splits to subdivide

In [10]:
df_train = df[df['tts'] == 'train']
df_test = df[df['tts'] == 'test']

# Shuffle to break any inherent ordering
df_train = df_train.sample(frac=1).reset_index(drop=True)
df_test = df_test.sample(frac=1).reset_index(drop=True)

print(df_train.shape)
print(df_test.shape)

display(df_train.head())
display(df_test.head())

(10532, 7)
(27050, 7)


,quantitative_function,seq,inferred_parents,num_effective_parents,frac_seq_explained,parent_contribution_mat,tts
0,0.557204,MSKGEELFTGVVPILVELDGDVHGHKFSVRGEGEGDADYGKLEIKI...,"[['EBFP'], ['OFPxm'], ['TagCFP', 'TagGFP2'], [...",4,[0.55042018 0.17226891 0.13865546 0.08823529],[[0 0 0 ... 1 1 1]\n [1 1 1 ... 0 0 0]\n [0 0 ...,train
1,1.352941,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATNGKLTLKF...,"[['EBFP'], ['moxVenus'], ['mAmetrine'], ['GFPh...",4,[0.4579832 0.22689076 0.13445378 0.10504201],[[1 1 1 ... 0 0 0]\n [0 0 0 ... 0 0 0]\n [0 0 ...,train
2,1.983960,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLEIKF...,"[['EGFP'], ['GFPxm191uv', 'OFPxm'], ['GFP-Tyr1...",4,[5.42016816e-01 1.93277312e-01 1.42857140e-01 ...,[[1 1 1 ... 1 1 1]\n [0 0 0 ... 0 0 0]\n [0 0 ...,train
3,1.157148,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,"[['EBFP'], ['muGFP', 'OFPxm'], ['W7', 'W2']]",3,[0.34873951 0.44117647 0.14285714],[[0 0 0 ... 0 0 0]\n [1 1 1 ... 0 0 0]\n [0 0 ...,train
4,0.800030,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATYGKLTLKF...,"[['W1C'], ['W2'], ['mT-Sapphire'], ['TagGFP'],...",6,[0.34873951 0.2184874 0.17647059 0.09243697 0...,[[0 0 0 ... 0 0 0]\n [0 0 0 ... 0 0 0]\n [0 0 ...,train


,quantitative_function,seq,inferred_parents,num_effective_parents,frac_seq_explained,parent_contribution_mat,tts
0,0.641645,MSKGEELFAGIVPILVELDGDVNGHKFSVSGEGEGDATHGKLTLKL...,"[['Aquamarine', 'EBFP'], ['Ypet'], ['aceGFP'],...",5,[0.23949581 0.21428572 0.15546219 0.13445378 0...,[[0 0 0 ... 1 1 1]\n [0 0 0 ... 1 1 1]\n [0 0 ...,test
1,1.983960,MSKGEEMFTGVVPILVELDGDVNGHKFSVRGEGEGDADYGKLEIKF...,"[['Dreiklang'], ['Clover'], ['T-Sapphire'], ['...",7,[3.02521022e-01 1.76470595e-01 1.42857145e-01 ...,[[0 0 0 ... 1 1 1]\n [0 0 0 ... 0 0 0]\n [0 0 ...,test
2,1.434310,MSKGEELFAGVVPILVELDGDVNGHKFSVSGEGEGDADYGKLEIKF...,"[['sfGFP_internal'], ['Clomeleon'], ['roGFP1-R...",4,[0.38235295 0.21428572 0.20168067 0.13445377],[[0 0 0 ... 1 1 1]\n [0 0 0 ... 0 0 0]\n [0 0 ...,test
3,1.983960,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,"[['Ypet'], ['Citrine'], ['roGFP1-R8', 'roGFP2'...",4,[0.38655463 0.28571429 0.18067226 0.07983193],[[1 1 1 ... 0 0 0]\n [1 1 1 ... 0 0 0]\n [0 0 ...,test
4,0.659753,MRKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,"[['PA-GFP', 'EBFP'], ['mCitrine'], ['Clover', ...",3,[0.52100841 0.25210084 0.16386554],[[0 0 0 ... 0 0 0]\n [0 0 0 ... 0 0 0]\n [0 0 ...,test


In [11]:
def split_df_into_thirds(df):
    # Note kfold split indices are sequential, but we've shuffled
    # the dataframes above. 
    kf = KFold(n_splits=3)
    split_indices = []
    for _, sp_idx in kf.split(df):
        split_indices.append(sp_idx)
        
    return [df.iloc[si] for si in split_indices]

### Subsplit training dataframes (data distributions)

In [12]:
subsplit_train_dfs = split_df_into_thirds(df_train)

for sdf in subsplit_train_dfs:
    display(sdf.head(n=2))
    print(sdf.shape)

,quantitative_function,seq,inferred_parents,num_effective_parents,frac_seq_explained,parent_contribution_mat,tts
0,0.557204,MSKGEELFTGVVPILVELDGDVHGHKFSVRGEGEGDADYGKLEIKI...,"[['EBFP'], ['OFPxm'], ['TagCFP', 'TagGFP2'], [...",4,[0.55042018 0.17226891 0.13865546 0.08823529],[[0 0 0 ... 1 1 1]\n [1 1 1 ... 0 0 0]\n [0 0 ...,train
1,1.352941,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATNGKLTLKF...,"[['EBFP'], ['moxVenus'], ['mAmetrine'], ['GFPh...",4,[0.4579832 0.22689076 0.13445378 0.10504201],[[1 1 1 ... 0 0 0]\n [0 0 0 ... 0 0 0]\n [0 0 ...,train


(3511, 7)


,quantitative_function,seq,inferred_parents,num_effective_parents,frac_seq_explained,parent_contribution_mat,tts
3511,1.98396,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLEIKF...,"[['EGFP'], ['SBFP2'], ['Aquamarine', 'EBFP'], ...",4,[0.40756304 0.21428572 0.16386554 0.13865546],[[1 1 1 ... 0 0 0]\n [1 1 1 ... 0 0 0]\n [1 1 ...,train
3512,1.98396,MSKGEELFTGVVPILAELDGDVNGHKFSVRGEGEGDATYGKLTLKF...,"[['SBFP2'], ['moxVenus'], ['GFP-151pyTyrCu'], ...",5,[0.36974791 0.18907564 0.18487395 0.09663865 0...,[[0 0 0 ... 0 0 0]\n [1 1 1 ... 0 0 0]\n [0 0 ...,train


(3511, 7)


,quantitative_function,seq,inferred_parents,num_effective_parents,frac_seq_explained,parent_contribution_mat,tts
7022,1.349349,MSGGEELFAGIVPVLIEMDGDVHGHKFSVSGEGEGDATNGKLTLKF...,"[['TagGFP2', 'PS-CFP2'], ['Aquamarine', 'EBFP'...",3,[0.60504202 0.18067227 0.12184874],[[1 1 1 ... 0 0 0]\n [0 0 0 ... 0 0 0]\n [0 0 ...,train
7023,0.731652,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATYGKLTLKF...,"[['H9'], ['mT-Sapphire'], ['EBFP2', 'moxBFP']]",3,[0.5462185 0.20588235 0.16386554],[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 ...,train


(3510, 7)


### Subsplit generalization sets

In [13]:
subsplit_test_dfs = split_df_into_thirds(df_test)

for sdf in subsplit_test_dfs:
    display(sdf.head(n=2))
    print(sdf.shape)

,quantitative_function,seq,inferred_parents,num_effective_parents,frac_seq_explained,parent_contribution_mat,tts
0,0.641645,MSKGEELFAGIVPILVELDGDVNGHKFSVSGEGEGDATHGKLTLKL...,"[['Aquamarine', 'EBFP'], ['Ypet'], ['aceGFP'],...",5,[0.23949581 0.21428572 0.15546219 0.13445378 0...,[[0 0 0 ... 1 1 1]\n [0 0 0 ... 1 1 1]\n [0 0 ...,test
1,1.983960,MSKGEEMFTGVVPILVELDGDVNGHKFSVRGEGEGDADYGKLEIKF...,"[['Dreiklang'], ['Clover'], ['T-Sapphire'], ['...",7,[3.02521022e-01 1.76470595e-01 1.42857145e-01 ...,[[0 0 0 ... 1 1 1]\n [0 0 0 ... 0 0 0]\n [0 0 ...,test


(9017, 7)


,quantitative_function,seq,inferred_parents,num_effective_parents,frac_seq_explained,parent_contribution_mat,tts
9017,1.911761,MSKGEELFTGVVPILIELDGDVNGHKFSVSGEGEGDADYGKLEIKF...,"[['Citrine'], ['Clover', 'mClover3'], ['TagYFP...",7,[2.10084049e-01 1.80672277e-01 1.38655466e-01 ...,[[0 0 0 ... 1 1 1]\n [0 0 0 ... 0 0 0]\n [0 0 ...,test
9018,1.983960,MSGGEELFAGIVPILVELDGDVNGHKFSVRGVGEGDATYGKLTLKF...,"[['Superfolder_GFP', 'Clover', 'sfGFP_internal...",7,[2.22689091e-01 1.97478999e-01 1.59663868e-01 ...,[[0 0 0 ... 0 0 0]\n [0 0 0 ... 0 0 0]\n [0 0 ...,test


(9017, 7)


,quantitative_function,seq,inferred_parents,num_effective_parents,frac_seq_explained,parent_contribution_mat,tts
18034,0.533398,MSKGAELFTGVVPILVELDGDVNGHKFSVGGEGEGDATYGKLTLKF...,"[['mEmerald'], ['D10'], ['Turquoise-GL'], ['Ta...",6,[2.31092452e-01 2.18487402e-01 1.63865547e-01 ...,[[0 0 0 ... 0 0 0]\n [0 0 0 ... 0 0 0]\n [0 0 ...,test
18035,1.983960,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,"[['Superfolder_GFP'], ['TagYFP'], ['J8VIQ3_9SP...",3,[0.46638657 0.24369748 0.22689075],[[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 ...,test


(9016, 7)


Some final checks. Make sure we're using all sequences, and assert that there is no overlap between any of the subsplits.

In [14]:
assert set(list(df['seq'])) == set(list(df_train['seq']) + list(df_test['seq']))

assert len( set(list(df_train['seq'])).intersection(set(list(df_test['seq']))) ) == 0

for i,adf in enumerate(subsplit_train_dfs):
    for j,bdf in enumerate(subsplit_train_dfs):
        
        if i != j:
            assert len( set(list(adf['seq'])).intersection(set(list(bdf['seq']))) ) == 0

for i,adf in enumerate(subsplit_test_dfs):
    for j,bdf in enumerate(subsplit_test_dfs):
        
        if i != j:
            assert len( set(list(adf['seq'])).intersection(set(list(bdf['seq']))) ) == 0      

## Export

In [15]:
train_subsplit_prefix = 'fp_homologs_data_dist_split_'
test_subsplit_prefix = 'fp_homologs_gen_split_'

In [16]:
for i,tdf in enumerate(subsplit_train_dfs):
    # Put in data distributions split dir
    ofile = os.path.join(paths.DATA_DISTRIBUTIONS_DIR, train_subsplit_prefix + str(i) + '.csv')
    tdf.to_csv(ofile, index=False)
    
    print(tdf.shape)
    print(ofile)
    print(data_io_utils.generate_md5_checksum(ofile))
    print()
    
    

(3511, 7)
/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/data_distributions/fp_homologs_data_dist_split_0.csv
Generate MD5 method
file_path: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/data_distributions/fp_homologs_data_dist_split_0.csv
506247c354303935139f2e26c96fb975

(3511, 7)
/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/data_distributions/fp_homologs_data_dist_split_1.csv
Generate MD5 method
file_path: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/data_distributions/fp_homologs_data_dist_split_1.csv
b1981c7996ac2f94a77697da4b05d138

(3510, 7)
/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/data_distributions/fp_homologs_data_dist_split_2.csv
Generate MD5 method
file_path: /content/drive/MyDrive/Colab Noteb

In [17]:
for i,tdf in enumerate(subsplit_test_dfs):
    # Put in generalization sets split dir
    ofile = os.path.join(paths.GEN_SETS_SPLITS_DIR, test_subsplit_prefix + str(i) + '.csv')
    tdf.to_csv(ofile, index=False)
    
    print(tdf.shape)
    print(ofile)
    print(data_io_utils.generate_md5_checksum(ofile))
    print()
    
    

(9017, 7)
/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/generalization_sets/fp_homologs_gen_split_0.csv
Generate MD5 method
file_path: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/generalization_sets/fp_homologs_gen_split_0.csv
5fbc611004b0658e6cefed6420f9164d

(9017, 7)
/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/generalization_sets/fp_homologs_gen_split_1.csv
Generate MD5 method
file_path: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/generalization_sets/fp_homologs_gen_split_1.csv
7cdd6d64d8bf4d3d10a89e75d7fe78ca

(9016, 7)
/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/tts_splits/generalization_sets/fp_homologs_gen_split_2.csv
Generate MD5 method
file_path: /content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysi

Manually verified these results are reproducible by running the notebook 2x top to bottom and checking MD5 checksums.

## Sync back up to S3

In [18]:
# Post publication note: Disabling sync to read-only bucket.
#data_io_utils.sync_local_path_to_s3(paths.TTS_SPLITS_DIR)